# Capital Allocation Analyst — the v2 Reliability Stack, on Gemini

A **from-scratch** capital-allocation analyst that applies the same reliability architecture as
[`code_assistant/claude_code_from_scratch_v2.ipynb`](../code_assistant/claude_code_from_scratch_v2.ipynb)
— re-pointed from **coding** to **capital allocation**, and running on **Gemini** through the
`google-genai` SDK. The coding notebook was the stepping-stone; this is the same machinery applied
to a non-coding domain (knowledge-management notebooks are next).

We own the whole perception→action loop: tool dispatch, planning, a bounded context window, and
the safety guards — while a hosted Gemini model does the reasoning.

### The stack (mirrors v2, phase for phase)

| Phase | v2 (coding) | here (capital allocation) |
|---|---|---|
| 1 Cognitive substrate | think / difficulty / **problem router** / self-consistency / verifier | same, re-pointed; the router (convergent/divergent/exploratory/structural) earns its keep here |
| 2 Tools | file/shell, lint-gated writes, test runner | **read-only BigQuery** (+ sample/profile/dry-run) + sandboxed **`calc`** with NPV/IRR/PI/RAROC helpers |
| 3 Tool loop + subagents | tool loop, isolated subagents | hand-rolled `agent_loop` over `generate_content` + `spawn_subagent` |
| 4 Hardening | architect/editor, self-refine, adversarial probe | same, re-pointed to red-teaming an allocation recommendation |
| 5 Planning + spec layer | SQLite DAG, bi-temporal memory, **pytest contract** | DAG + bi-temporal memory + **`verify_analysis`: independent recomputation** |
| 6 Context engineering | trim → distill → reinject → consolidate | same, over Gemini `Content` turns (supersedes simple compaction) |
| 7 MCP + 5 subagents | Planner/Implementer/Tester/Reviewer/ReportWriter | Planner/**Analyst**/**Verifier**/Reviewer/ReportWriter |

**Tools:** read-only BigQuery (`list_datasets`, `list_tables`, `get_table_info`, `get_table_sample`,
`profile_column`, `execute_sql`, `dry_run_cost`) + `calc`. The only way this agent touches the world
is SELECT queries and arithmetic — no shell, no writes.

**Skills:** domain playbooks under `fin_sandbox/skills/` (`raroc`, `npv-irr-conventions`) are loaded
on demand via `load_skill`; their `calc` helpers (`pi`, `payback`, `raroc`, `economic_profit`, …) are
built in.

**Safety:** every SQL statement is gated to `SELECT`/`WITH`, dry-run estimates bytes scanned, and a
`maximum_bytes_billed` cap aborts runaway scans. Point it at a warehouse only with least-privilege
credentials; every query and model call bills your own GCP project.

---

### Setup

```bash
pip install google-genai google-cloud-bigquery

# Option A — Vertex AI (one ADC drives BOTH Gemini and BigQuery)
gcloud auth application-default login
export GOOGLE_CLOUD_PROJECT=your-gcp-project
export GOOGLE_CLOUD_LOCATION=us-central1
export GOOGLE_GENAI_USE_VERTEXAI=TRUE

# Option B — Gemini Developer API key for the MODEL (BigQuery still needs GCP creds)
export GEMINI_API_KEY=your-key
```

In [ ]:
"""
Imports + logging.

One logger named "fin" with per-subsystem children (loop, gemini, tool, sub,
todo, compact, chat) so you can see exactly what the agent is doing. Flip
AGENT_LOG_LEVEL=DEBUG to see thoughts and full payloads.
"""
from __future__ import annotations

import json
import logging
import math
import os
import re
import sys
import time
import uuid
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

# google-genai is the unified SDK for both Vertex AI and the Gemini Dev API.
from google import genai
from google.genai import types


AGENT_LOG_LEVEL = os.environ.get("AGENT_LOG_LEVEL", "INFO").upper()


class _Fmt(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        short = record.name.split(".", 1)[1] if "." in record.name else record.name
        return f"[{record.levelname:5s}] {short:14s} | {record.getMessage()}"


_handler = logging.StreamHandler(sys.stdout)
_handler.setFormatter(_Fmt())

log = logging.getLogger("fin")
log.handlers.clear()
log.addHandler(_handler)
log.setLevel(AGENT_LOG_LEVEL)
log.propagate = False

log_loop    = logging.getLogger("fin.loop")
log_gemini  = logging.getLogger("fin.gemini")
log_tool    = logging.getLogger("fin.tool")
log_sub     = logging.getLogger("fin.subagent")
log_todo    = logging.getLogger("fin.todo")
log_compact = logging.getLogger("fin.compact")
log_chat    = logging.getLogger("fin.chat")

log.info(f"Logger initialised at level {AGENT_LOG_LEVEL}")

In [ ]:
"""
Configuration. All knobs live here.

Backend selection:
  - If GEMINI_API_KEY is set  -> Gemini Developer API for the MODEL.
  - Otherwise                 -> Gemini via Vertex AI (needs GOOGLE_CLOUD_PROJECT
                                 + Application Default Credentials).
BigQuery always uses ADC regardless of which model backend you pick.
"""

# --- Model per agent role ----------------------------------------------------
# Lead drives planning + tool orchestration (needs the strongest reasoning).
# Subagent runs isolated, throwaway exploration -> a faster/cheaper model.
# Summariser only condenses old turns -> the cheap model is plenty.
MODELS = {
    "lead":       os.environ.get("FIN_LEAD_MODEL",       "gemini-2.5-pro"),
    "subagent":   os.environ.get("FIN_SUBAGENT_MODEL",   "gemini-2.5-flash"),
    "summarizer": os.environ.get("FIN_SUMMARIZER_MODEL", "gemini-2.5-flash"),
}

# --- Backend / auth ----------------------------------------------------------
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
USE_VERTEX = not bool(GEMINI_API_KEY)
if USE_VERTEX:
    os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "TRUE")
PROJECT  = os.environ.get("GOOGLE_CLOUD_PROJECT", "")          # set this for Vertex/BQ
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")
os.environ.setdefault("GOOGLE_CLOUD_LOCATION", LOCATION)

# BigQuery project for queries (defaults to PROJECT / ADC project).
BQ_PROJECT = os.environ.get("FIN_BQ_PROJECT", "")

# --- Gemini 2.5 "thinking" budget (tokens). -1 = let the model decide. -------
THINKING_BUDGET = int(os.environ.get("FIN_THINKING_BUDGET", "-1"))

# --- BigQuery cost / context hygiene -----------------------------------------
BQ_MAX_RESULT_ROWS  = int(os.environ.get("BQ_MAX_RESULT_ROWS", "100"))
BQ_MAX_BYTES_BILLED = int(os.environ.get("BQ_MAX_BYTES_BILLED", str(2 * 1024**3)))  # 2 GiB

# --- Agent limits ------------------------------------------------------------
MAX_TOOL_OUTPUT    = 20_000   # truncate big tool outputs to protect context
MAX_TURNS          = 25       # hard cap on loop iterations per query
COMPRESS_THRESHOLD = 60_000   # chars across all turns -> trigger compaction
KEEP_RECENT        = 6        # turns kept verbatim during compaction

# --- Local state (planning / memory) -----------------------------------------
WORKDIR = Path(os.environ.get("FIN_WORKDIR", str(Path.cwd() / "fin_sandbox"))).resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)
SKILLS_DIR  = Path(os.environ.get("FIN_SKILLS_DIR", str(WORKDIR / "skills")))
TODO_FILE   = WORKDIR / ".fin_todo.json"
TASKS_FILE  = WORKDIR / ".fin_tasks.json"
MEMORY_FILE = WORKDIR / ".fin_memory.md"

log.info(f"backend     = {'Vertex AI' if USE_VERTEX else 'Gemini Dev API'}")
log.info(f"MODELS      = {MODELS}")
log.info(f"PROJECT     = {PROJECT or '(unset -- set GOOGLE_CLOUD_PROJECT)'}")
log.info(f"WORKDIR     = {WORKDIR}")

In [ ]:
"""
Clients + healthchecks.

`client`  -> google-genai client (Vertex or Dev API), used for every model call.
`bq_client()` -> lazy google-cloud-bigquery client (created on first use so the
                 notebook imports cleanly even before you authenticate).

gemini_generate() is the single thin wrapper every loop call goes through:
no retry, no streaming -- errors bubble up so the loop can decide what to do.
"""

# --- Gemini client -----------------------------------------------------------
if USE_VERTEX:
    client = genai.Client(vertexai=True, project=(PROJECT or None), location=LOCATION)
else:
    client = genai.Client(api_key=GEMINI_API_KEY)


def gemini_generate(*, model: str, contents: list, config: "types.GenerateContentConfig",
                    label: str = "lead"):
    """One model call. Logs latency + token usage; returns the raw response."""
    n = len(contents)
    log_gemini.info(f"[{label}] -> {model}  turns={n}")
    t0 = time.time()
    resp = client.models.generate_content(model=model, contents=contents, config=config)
    dt = time.time() - t0
    um = getattr(resp, "usage_metadata", None)
    toks = getattr(um, "total_token_count", None) if um else None
    log_gemini.info(f"[{label}] <- {model}  {dt:5.1f}s  tokens={toks if toks is not None else '?'}")
    return resp


def gemini_healthcheck() -> bool:
    """Confirm we can reach the model and get text back."""
    try:
        r = client.models.generate_content(
            model=MODELS["lead"],
            contents="Reply with the single word: OK",
            config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=512),
        )
        txt = (getattr(r, "text", None) or "").strip()
        log_gemini.info(f"healthcheck OK -- {MODELS['lead']} replied {txt[:40]!r}")
        return True
    except Exception as e:
        log_gemini.error(f"healthcheck FAILED: {e}")
        return False


# --- BigQuery client (lazy) --------------------------------------------------
_bq = None


def bq_client():
    """Create (once) and return a BigQuery client using ADC."""
    global _bq
    if _bq is None:
        from google.cloud import bigquery  # imported lazily
        _bq = bigquery.Client(project=(BQ_PROJECT or PROJECT or None))
        log.info(f"BigQuery client ready (project={_bq.project})")
    return _bq


def bigquery_healthcheck() -> bool:
    try:
        ds = list(bq_client().list_datasets(max_results=1))
        log.info(f"BigQuery healthcheck OK -- project={bq_client().project}, datasets visible={len(ds)>=0}")
        return True
    except Exception as e:
        log.error(f"BigQuery healthcheck FAILED: {e}")
        return False


log.info("Clients ready. Run gemini_healthcheck() / bigquery_healthcheck() once authenticated.")

## Phase 1 — Cognitive substrate

The article's first move: stop treating the model as a one-shot oracle. Four backend-neutral
patterns, ported from v2 and re-pointed at capital allocation:

- **Thinking channel** (`think_then_answer`) — separate deliberation from the answer; Gemini emits
  thought parts (`part.thought == True`) which we split out and never feed back.
- **Compute-adaptive effort** (`adaptive_think`) — two cheap classifiers shape the response:
  `estimate_difficulty` sizes the token *budget*, and `classify_problem` picks the *strategy*
  (convergent → self-consistency, divergent → verifier-ranked candidates, exploratory → one wide
  pass, structural → decompose-then-assemble). Unlike spec-driven coding, capital-allocation
  questions span all four types, so the router does real work here.
- **Self-consistency** — sample *k* answers, take the majority.
- **Verifier asymmetry** (`verifier_score` + `asymmetric_solve`) — a cheap generator proposes, a
  strong verifier ranks. Checking is cheaper than producing.

In [ ]:
"""
Phase 1 -- cognitive substrate (ported from claude_code_from_scratch_v2.ipynb).

Backend-neutral reasoning patterns, re-pointed from coding to capital allocation and
re-expressed on the Gemini wrapper:
  - think_then_answer : separate deliberation from answer (Gemini's native thought parts).
  - estimate_difficulty + adaptive_think : the article's TWO adaptive-compute axes --
      difficulty sizes the token budget, classify_problem picks the solving strategy.
  - self_consistency  : sample k answers, take the majority.
  - verifier asymmetry: a cheap generator proposes, a strong verifier ranks.

These call only the thin Gemini helpers below, so the whole substrate is domain-neutral --
the same code carries into the knowledge-management notebooks next.
"""
from concurrent.futures import ThreadPoolExecutor
from collections import Counter
from dataclasses import dataclass

# --- thin generation helpers over gemini_generate ---------------------------
def _user(text: str) -> "types.Content":
    return types.Content(role="user", parts=[types.Part.from_text(text=text)])


def parse_json(text: str):
    """Tolerant JSON parse: strip ```json fences, then fall back to the first {...} span."""
    t = (text or "").strip()
    if t.startswith("```"):
        t = t.split("```")[1]
        if t.lstrip().startswith("json"):
            t = t.lstrip()[4:]
    try:
        return json.loads(t)
    except Exception:
        m = re.search(r"\{.*\}", t, re.DOTALL)
        if m:
            return json.loads(m.group(0))
        raise


def gen_json(prompt: str, *, model: str = None, system: str = "",
             max_tokens: int = 1024, label: str = "json") -> Any:
    """One structured call. response_mime_type=JSON + thinking off -> fast, parseable."""
    model = model or MODELS["summarizer"]
    cfg = types.GenerateContentConfig(
        system_instruction=system or None, temperature=0.0,
        max_output_tokens=max_tokens, response_mime_type="application/json",
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    )
    resp = gemini_generate(model=model, contents=[_user(prompt)], config=cfg, label=label)
    return parse_json(getattr(resp, "text", None) or "")


@dataclass
class ThoughtfulResponse:
    thinking: str
    answer: str
    output_tokens: int


SUBSTRATE_SYSTEM = (
    "You are a careful, senior capital-allocation analyst. You think before you answer, "
    "you name your assumptions (discount/hurdle rate, horizon, currency, capital budget) "
    "before you rely on them, and you never assert a figure you have not actually computed. "
    "Checking is cheaper than producing: prefer answers you can re-derive."
)


def think_then_answer(query: str, *, model: str = None, temperature: float = 0.3,
                      max_tokens: int = 2048, system: str = SUBSTRATE_SYSTEM) -> ThoughtfulResponse:
    """Free-text call that keeps Gemini's thinking channel separate from the answer."""
    model = model or MODELS["subagent"]
    cfg = types.GenerateContentConfig(
        system_instruction=system or None, temperature=temperature, max_output_tokens=max_tokens,
        thinking_config=types.ThinkingConfig(thinking_budget=THINKING_BUDGET, include_thoughts=True),
    )
    resp = gemini_generate(model=model, contents=[_user(query)], config=cfg, label="think")
    cand = (resp.candidates or [None])[0]
    thoughts, answer = [], []
    for p in (getattr(getattr(cand, "content", None), "parts", None) or []):
        if getattr(p, "thought", False) and getattr(p, "text", None):
            thoughts.append(p.text)
        elif getattr(p, "text", None):
            answer.append(p.text)
    um = getattr(resp, "usage_metadata", None)
    toks = getattr(um, "candidates_token_count", 0) if um else 0
    return ThoughtfulResponse("".join(thoughts).strip(), "".join(answer).strip(), toks or 0)


# --- axis 1: difficulty -> token budget --------------------------------------
def estimate_difficulty(query: str) -> str:
    """Cheap classifier sizing how much deliberation the question deserves."""
    try:
        d = gen_json(
            'Classify the difficulty of this capital-allocation question. Output JSON: '
            '{"difficulty": one of "trivial","easy","medium","hard","extreme"}.\n\n' + query,
            model=MODELS["summarizer"], max_tokens=64, label="difficulty")
        return str(d.get("difficulty", "medium")).lower()
    except Exception:
        return "medium"


THINKING_BUDGETS = {"trivial": 256, "easy": 512, "medium": 1500, "hard": 3000, "extreme": 6000}


# --- axis 2: problem TYPE -> solving strategy (the router carried over from v2) ----
PROBLEM_TYPES = ["convergent", "divergent", "exploratory", "structural"]


def classify_problem(query: str) -> dict:
    """Classify the *kind* of problem (orthogonal to difficulty). One cheap JSON call.

    For capital allocation the type genuinely varies, so the router earns its keep:
      convergent  -> one defensible number (NPV/IRR/RAROC of a given stream); trust on agreement.
      divergent   -> many valid answers (which projects to fund, hedging strategies); rank candidates.
      exploratory -> open-ended (what drives this book's returns?); one wide, higher-budget pass.
      structural  -> decompose-then-assemble (build the multi-unit allocation model); plan first.
    """
    try:
        d = gen_json(
            "Classify the KIND of problem (not its difficulty). Output JSON: "
            '{"type": one of "convergent","divergent","exploratory","structural", '
            '"reason": str}.\n'
            "convergent  = one correct/defensible answer (a metric, a calculation).\n"
            "divergent   = many valid answers (allocations, strategies, recommendations).\n"
            "exploratory = open-ended or under-specified (drivers, implications, 'what if').\n"
            "structural  = needs decomposition into parts before answering (multi-unit builds).\n\n"
            + query, model=MODELS["summarizer"], max_tokens=128, label="classify")
        ptype = str(d.get("type", "convergent")).lower()
        if ptype not in PROBLEM_TYPES:
            ptype = "convergent"
        return {"type": ptype, "reason": str(d.get("reason", ""))}
    except Exception:
        return {"type": "convergent", "reason": "fallback (classifier parse failed)"}


TYPE_STRATEGY = {
    "convergent":  "self_consistency",   # majority vote over independent samples
    "divergent":   "asymmetric_solve",   # diverse candidates + a strong verifier ranks them
    "exploratory": "wide_pass",          # one higher-temperature, higher-budget pass
    "structural":  "decompose",          # plan-first single pass
}


# --- self-consistency + verifier asymmetry -----------------------------------
def self_consistency(query: str, k: int = 3, model: str = None) -> dict:
    """Sample k answers in parallel, return the majority bucket (first 60 chars)."""
    model = model or MODELS["subagent"]
    with ThreadPoolExecutor(max_workers=k) as ex:
        samples = list(ex.map(
            lambda _: think_then_answer(query, model=model, temperature=0.7).answer, range(k)))
    keys = [s[:60].lower() for s in samples]
    winner_key, votes = Counter(keys).most_common(1)[0]
    winner = next(s for s in samples if s[:60].lower() == winner_key)
    return {"winner": winner, "votes": votes, "k": k,
            "agreement": votes / k, "all_samples": samples}


VERIFIER_SYSTEM = (
    "You are a careful, structured verifier of capital-allocation reasoning. Score the "
    "candidate 1-10 (10 = perfect, 1 = unusable) on CORRECTNESS of the figures and logic, "
    "not style. Penalise uncited hurdle rates, mixed currencies, and arithmetic done in the "
    'head. Output JSON: {"score": int, "reason": str (one sentence)}.'
)


def verifier_score(question: str, candidate: str, verifier_model: str = None) -> dict:
    verifier_model = verifier_model or MODELS["lead"]
    try:
        return gen_json(f"QUESTION:\n{question}\n\nCANDIDATE:\n{candidate}",
                        model=verifier_model, system=VERIFIER_SYSTEM, max_tokens=300, label="verify")
    except Exception as e:
        return {"score": 0, "reason": f"verifier parse error: {e}"}


def asymmetric_solve(query: str, n_candidates: int = 3) -> dict:
    """Verifier asymmetry: cheap generator (n candidates) + one strong ranking call."""
    with ThreadPoolExecutor(max_workers=n_candidates) as ex:
        candidates = list(ex.map(
            lambda _: think_then_answer(query, model=MODELS["subagent"], temperature=0.7).answer,
            range(n_candidates)))
    rank_prompt = (
        'Rank these candidate analyses best-to-worst. Output JSON: '
        '{"ranking": [{"rank": 1, "index": 0, "reason": "..."}, ...]}.\n\n'
        + "\n\n".join(f"CANDIDATE {i}:\n{c}" for i, c in enumerate(candidates)))
    try:
        ranking = gen_json(rank_prompt, model=MODELS["lead"], max_tokens=600, label="rank")["ranking"]
        idx = ranking[0]["index"]
        return {"winner": candidates[idx], "winner_reason": ranking[0].get("reason", ""),
                "ranking": ranking}
    except Exception:
        return {"winner": candidates[0], "winner_reason": "fallback (ranking parse failed)",
                "ranking": []}


def adaptive_think(query: str, route: bool = True) -> dict:
    """Compute-adaptive effort on BOTH axes: difficulty -> budget, problem type -> strategy.
    route=False falls back to the budget-only single pass."""
    difficulty = estimate_difficulty(query)
    budget = THINKING_BUDGETS.get(difficulty, 1500)
    problem = classify_problem(query) if route else {"type": "convergent", "reason": "routing off"}
    ptype = problem["type"]
    strategy = TYPE_STRATEGY.get(ptype, "self_consistency") if route else "single_pass"
    log.info(f"[adaptive] difficulty={difficulty} type={ptype} -> budget {budget}, strategy {strategy}")

    if strategy == "self_consistency":
        r = self_consistency(query, k=3)
        answer, extra = r["winner"], {"agreement": r["agreement"], "votes": r["votes"]}
    elif strategy == "asymmetric_solve":
        r = asymmetric_solve(query, n_candidates=3)
        answer, extra = r["winner"], {"winner_reason": r["winner_reason"]}
    elif strategy == "decompose":
        r = think_then_answer("Break this into parts, solve each, then assemble the result:\n\n"
                              + query, max_tokens=budget)
        answer, extra = r.answer, {"actual_tokens": r.output_tokens}
    elif strategy == "wide_pass":
        r = think_then_answer(query, temperature=0.7, max_tokens=max(budget, 3000))
        answer, extra = r.answer, {"actual_tokens": r.output_tokens}
    else:  # single_pass
        r = think_then_answer(query, max_tokens=budget)
        answer, extra = r.answer, {"actual_tokens": r.output_tokens}

    return {"difficulty": difficulty, "type": ptype, "reason": problem["reason"],
            "budget": budget, "strategy": strategy, "answer": answer, **extra}


log.info("Phase 1 substrate ready: think_then_answer, adaptive_think (difficulty + problem router), "
         "self_consistency, verifier_score, asymmetric_solve")

## Phase 2 — Tools: read-only BigQuery + sandboxed finance math

The agent's hands. Two families, both safe by construction:

- **`calc`** — a sandboxed evaluator (`__builtins__` stripped) exposing every `math.*` function
  plus finance helpers: `npv`, `irr`, `fv`/`pv`/`pmt`, and (new) `pi`, `payback`, `disc_payback`,
  `eaa`, `expected_loss`, `raroc`, `economic_profit`, `cagr` — the exact primitives the `raroc`
  and `npv-irr-conventions` skills call for.
- **Read-only BigQuery** — `list_datasets`, `list_tables`, `get_table_info`, `execute_sql`, plus
  the new `get_table_sample` (see real values), `profile_column` (grain/range in one pass), and
  `dry_run_cost` (size a query before paying). Every statement is gated to `SELECT`/`WITH`,
  dry-run-estimated, and capped at `maximum_bytes_billed`.

In [ ]:
"""
`calc` -- sandboxed financial-math tool (ported verbatim from the ADK notebook).

The model passes a Python expression; we eval it with __builtins__ stripped and
a curated namespace (every math.* function + Excel-style finance helpers).
"""

def _npv(rate, cashflows):
    return sum(cf / (1.0 + rate) ** i for i, cf in enumerate(cashflows))


def _irr(cashflows, guess: float = 0.1):
    lo, hi = -0.9999, 10.0
    f_lo, f_hi = _npv(lo, cashflows), _npv(hi, cashflows)
    if f_lo * f_hi <= 0:                       # bracketed -> bisection
        for _ in range(200):
            mid = (lo + hi) / 2.0
            f_mid = _npv(mid, cashflows)
            if abs(f_mid) < 1e-10:
                return mid
            if f_lo * f_mid < 0:
                hi = mid
            else:
                lo, f_lo = mid, f_mid
        return (lo + hi) / 2.0
    r = guess                                  # not bracketed -> Newton
    for _ in range(100):
        base = _npv(r, cashflows)
        deriv = (_npv(r + 1e-6, cashflows) - base) / 1e-6
        if deriv == 0:
            break
        step = base / deriv
        r -= step
        if abs(step) < 1e-10:
            return r
    return r


def _fv(rate, nper, pmt, pv=0.0):
    if rate == 0:
        return -(pv + pmt * nper)
    return -(pv * (1 + rate) ** nper + pmt * ((1 + rate) ** nper - 1) / rate)


def _pv(rate, nper, pmt, fv=0.0):
    if rate == 0:
        return -(fv + pmt * nper)
    return -(fv + pmt * ((1 + rate) ** nper - 1) / rate) / (1 + rate) ** nper


def _pmt(rate, nper, pv, fv=0.0):
    if rate == 0:
        return -(pv + fv) / nper
    return -(pv * (1 + rate) ** nper + fv) * rate / ((1 + rate) ** nper - 1)


def _payback(cashflows):
    """Undiscounted payback period (in periods); None if never recovered."""
    cum = 0.0
    for i, cf in enumerate(cashflows):
        cum += cf
        if cum >= 0:
            if i == 0:
                return 0.0
            prev = cum - cf
            return (i - 1) + (-prev) / cf if cf != 0 else float(i)
    return None


def _disc_payback(rate, cashflows):
    """Discounted payback period (in periods); None if never recovered."""
    cum = 0.0
    for i, cf in enumerate(cashflows):
        d = cf / (1.0 + rate) ** i
        cum += d
        if cum >= 0:
            if i == 0:
                return 0.0
            prev = cum - d
            return (i - 1) + (-prev) / d if d != 0 else float(i)
    return None


def _pi(rate, cashflows):
    """Profitability index = PV(inflows) / |initial outlay|. Outlay = -cashflows[0]."""
    outlay = -cashflows[0]
    if outlay <= 0:
        raise ValueError("pi() expects cashflows[0] < 0 (an initial outlay)")
    pv_in = sum(cf / (1.0 + rate) ** i for i, cf in enumerate(cashflows) if i > 0)
    return pv_in / outlay


def _eaa(rate, cashflows):
    """Equivalent annual annuity of an NPV over n = len-1 periods."""
    n = len(cashflows) - 1
    npv = _npv(rate, cashflows)
    if n <= 0:
        raise ValueError("eaa() needs at least one period after t=0")
    if rate == 0:
        return npv / n
    annuity_factor = (1 - (1 + rate) ** (-n)) / rate
    return npv / annuity_factor


def _expected_loss(pd, lgd, ead):
    """EL = PD x LGD x EAD (the RAROC risk adjustment)."""
    return pd * lgd * ead


def _raroc(revenue, funding_cost, opex, expected_loss, econ_capital):
    """RAROC = risk-adjusted net income / economic capital (see the raroc skill)."""
    if econ_capital == 0:
        raise ValueError("raroc() economic capital is zero")
    return (revenue - funding_cost - opex - expected_loss) / econ_capital


def _economic_profit(raroc, hurdle, econ_capital):
    """EVA-style spread x capital: (RAROC - hurdle) x economic_capital."""
    return (raroc - hurdle) * econ_capital


def _cagr(begin, end, periods):
    """Compound annual growth rate."""
    if begin <= 0 or periods <= 0:
        raise ValueError("cagr() needs begin>0 and periods>0")
    return (end / begin) ** (1.0 / periods) - 1.0


def tool_calc(expression: str) -> dict:
    """Evaluate a Python/finance math expression. Namespace exposes every
    math.* function plus: npv(rate, cashflows), irr(cashflows),
    fv/pv/pmt, pi(rate,cf), payback(cf), disc_payback(rate,cf), eaa(rate,cf),
    expected_loss(pd,lgd,ead), raroc(rev,fund,opex,el,cap), economic_profit(raroc,hurdle,cap), cagr(begin,end,n)."""
    log_tool.info(f"[calc] {expression[:160]}")
    ns = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    ns.update({
        "abs": abs, "round": round, "min": min, "max": max, "sum": sum,
        "len": len, "pow": pow, "range": range, "sorted": sorted,
        "npv": _npv, "irr": _irr, "fv": _fv, "pv": _pv, "pmt": _pmt,
        "payback": _payback, "disc_payback": _disc_payback, "pi": _pi, "eaa": _eaa,
        "expected_loss": _expected_loss, "raroc": _raroc,
        "economic_profit": _economic_profit, "cagr": _cagr,
    })
    try:
        result = eval(expression, {"__builtins__": {}}, ns)  # sandboxed
        return {"status": "ok", "result": str(result)}
    except Exception as e:
        return {"status": "error", "error": f"{e}  (expression: {expression!r})"}


# quick self-check (no agent / no GCP needed)
print(tool_calc("npv(0.10, [-1000, 300, 400, 500, 600])"))
print(tool_calc("pmt(0.05/12, 360, 400000)"))

In [ ]:
"""
Read-only BigQuery tools -- the hand-rolled equivalent of ADK's BigQueryToolset
with WriteMode.BLOCKED.

  list_datasets()            -> dataset ids in the project
  list_tables(dataset_id)    -> table ids in a dataset
  get_table_info(table_ref)  -> schema + row count + description
  execute_sql(sql)           -> rows, gated to SELECT/WITH with a dry-run cost guard

Safety: execute_sql refuses anything that is not a single SELECT/WITH statement,
dry-runs it to estimate bytes scanned, aborts if that exceeds BQ_MAX_BYTES_BILLED,
and caps returned rows at BQ_MAX_RESULT_ROWS.
"""

_DML_DDL = ("insert", "update", "delete", "merge", "drop", "create", "alter",
            "truncate", "grant", "revoke", "call", "export", "load", "replace")


def _strip_sql(sql: str) -> str:
    s = re.sub(r"/\*.*?\*/", " ", sql, flags=re.S)        # block comments
    s = re.sub(r"--[^\n]*", " ", s)                       # line comments
    return s.strip().rstrip(";").strip()


def _is_read_only(sql: str) -> bool:
    s = _strip_sql(sql).lstrip("(").strip().lower()
    if not (s.startswith("select") or s.startswith("with")):
        return False
    first_tokens = set(re.findall(r"[a-z_]+", s)[:1])     # leading keyword only
    if first_tokens & set(_DML_DDL):
        return False
    # reject a statement-initial DML/DDL keyword appearing after a semicolon split
    return ";" not in _strip_sql(sql)


def tool_list_datasets() -> dict:
    ds = [d.dataset_id for d in bq_client().list_datasets()]
    log_tool.info(f"[bq.list_datasets] {len(ds)} datasets")
    return {"status": "ok", "project": bq_client().project, "datasets": ds}


def tool_list_tables(dataset_id: str) -> dict:
    tabs = [t.table_id for t in bq_client().list_tables(dataset_id)]
    log_tool.info(f"[bq.list_tables] {dataset_id}: {len(tabs)} tables")
    return {"status": "ok", "dataset": dataset_id, "tables": tabs}


def tool_get_table_info(table_ref: str) -> dict:
    log_tool.info(f"[bq.get_table_info] {table_ref}")
    try:
        t = bq_client().get_table(table_ref)  # 'dataset.table' or 'project.dataset.table'
    except Exception as e:
        return {"status": "error", "error": f"{e}"}
    schema = [{"name": f.name, "type": f.field_type, "mode": f.mode,
               "description": f.description or ""} for f in t.schema]
    return {"status": "ok", "table": str(t.reference), "num_rows": t.num_rows,
            "description": t.description or "", "schema": schema}


def tool_execute_sql(sql: str) -> dict:
    log_tool.info(f"[bq.execute_sql] {sql[:160]}")
    if not _is_read_only(sql):
        return {"status": "error",
                "error": "Refused: only a single read-only SELECT/WITH statement is allowed."}
    from google.cloud import bigquery
    try:
        dry = bq_client().query(
            sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
        scanned = dry.total_bytes_processed or 0
    except Exception as e:
        return {"status": "error", "error": f"dry-run failed: {e}"}
    gb = scanned / 1024**3
    if scanned > BQ_MAX_BYTES_BILLED:
        return {"status": "error",
                "error": f"Refused: query would scan {gb:.2f} GiB > cap "
                         f"{BQ_MAX_BYTES_BILLED/1024**3:.2f} GiB. Add filters/aggregation."}
    try:
        job = bq_client().query(
            sql, job_config=bigquery.QueryJobConfig(
                maximum_bytes_billed=BQ_MAX_BYTES_BILLED, use_query_cache=True))
        rows = [dict(r) for r in job.result(max_results=BQ_MAX_RESULT_ROWS)]
    except Exception as e:
        return {"status": "error", "error": f"query failed: {e}"}
    rows = json.loads(json.dumps(rows, default=str))     # coerce dates/decimals
    log_tool.info(f"[bq.execute_sql] {len(rows)} rows, ~{gb:.4f} GiB scanned")
    return {"status": "ok", "gib_scanned": round(gb, 4), "row_count": len(rows), "rows": rows}


log.info("BigQuery tools defined: list_datasets, list_tables, get_table_info, execute_sql")


# --- expanded read-only exploration tools (Phase 2 additions) ----------------
def tool_get_table_sample(table_ref: str, n: int = 5) -> dict:
    """Return a few sample rows so the model sees real values, not just a schema.
    Goes through the same read-only / cost-guarded path as execute_sql."""
    n = max(1, min(int(n), 20))
    log_tool.info(f"[bq.get_table_sample] {table_ref} n={n}")
    return tool_execute_sql(f"SELECT * FROM `{table_ref}` LIMIT {n}")


def tool_profile_column(table_ref: str, column: str) -> dict:
    """Cheap one-pass profile of a column: row count, distinct, nulls, min, max.
    Use to confirm a column's grain / range before trusting it in a metric."""
    log_tool.info(f"[bq.profile_column] {table_ref}.{column}")
    sql = (f"SELECT COUNT(*) AS n, COUNT(DISTINCT `{column}`) AS distinct_n, "
           f"COUNTIF(`{column}` IS NULL) AS null_n, "
           f"MIN(`{column}`) AS min_v, MAX(`{column}`) AS max_v "
           f"FROM `{table_ref}`")
    return tool_execute_sql(sql)


def tool_dry_run_cost(sql: str) -> dict:
    """Estimate bytes a query would scan WITHOUT running it -- lets the model
    size a query before paying for it."""
    log_tool.info(f"[bq.dry_run_cost] {sql[:120]}")
    if not _is_read_only(sql):
        return {"status": "error", "error": "Refused: read-only SELECT/WITH only."}
    from google.cloud import bigquery
    try:
        dry = bq_client().query(
            sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
        scanned = dry.total_bytes_processed or 0
        return {"status": "ok", "gib_estimated": round(scanned / 1024**3, 4),
                "within_cap": scanned <= BQ_MAX_BYTES_BILLED,
                "cap_gib": round(BQ_MAX_BYTES_BILLED / 1024**3, 2)}
    except Exception as e:
        return {"status": "error", "error": f"dry-run failed: {e}"}


log.info("Expanded BQ tools defined: get_table_sample, profile_column, dry_run_cost")

In [ ]:
"""
Tool schemas (Gemini FunctionDeclarations) + base dispatch map.

Gemini takes tools as types.Tool(function_declarations=[FunctionDeclaration...]).
We keep our familiar JSON-schema-ish dicts and convert them with _schema/_fn so
the declarations read like the other from-scratch notebooks.

Two registries:
  DECLS_BASE / DISPATCH_BASE -- data + math tools (BigQuery + calc). Subagents
    get exactly this set.
  DECLS_LEAD / DISPATCH_LEAD -- later cells extend these with spawn_subagent,
    todo_*, task_*, list/load_skill for the lead agent.
"""

_TYPE = {"string": types.Type.STRING, "integer": types.Type.INTEGER,
         "number": types.Type.NUMBER, "boolean": types.Type.BOOLEAN,
         "array": types.Type.ARRAY, "object": types.Type.OBJECT}


def _schema(d: Dict[str, Any]) -> "types.Schema":
    kw: Dict[str, Any] = {"type": _TYPE[d["type"]]}
    if "description" in d:
        kw["description"] = d["description"]
    if "enum" in d:
        kw["enum"] = d["enum"]
    if d["type"] == "array" and "items" in d:
        kw["items"] = _schema(d["items"])
    if d["type"] == "object" and "properties" in d:
        kw["properties"] = {k: _schema(v) for k, v in d["properties"].items()}
        if "required" in d:
            kw["required"] = d["required"]
    return types.Schema(**kw)


def _fn(name: str, description: str,
        properties: Dict[str, Any] = None, required: List[str] = None) -> "types.FunctionDeclaration":
    """Build one FunctionDeclaration. No-arg tools pass properties=None."""
    params = None
    if properties:
        params = types.Schema(type=types.Type.OBJECT,
                              properties={k: _schema(v) for k, v in properties.items()},
                              required=required or [])
    return types.FunctionDeclaration(name=name, description=description, parameters=params)


DECLS_BASE: List["types.FunctionDeclaration"] = [
    _fn("calc",
        "Evaluate a math/finance expression. Use for ALL arithmetic. Helpers: "
        "npv, irr, fv/pv/pmt, pi, payback, disc_payback, eaa, raroc, economic_profit, "
        "expected_loss, cagr, plus every math.* function.",
        {"expression": {"type": "string", "description": "A single Python expression."}},
        ["expression"]),
    _fn("list_datasets",
        "List dataset ids available in the BigQuery project. Call this first to orient."),
    _fn("list_tables",
        "List table ids inside a dataset.",
        {"dataset_id": {"type": "string"}}, ["dataset_id"]),
    _fn("get_table_info",
        "Get a table's schema, row count and description. Use BEFORE writing SQL "
        "so you never invent column names.",
        {"table_ref": {"type": "string",
                       "description": "'dataset.table' or 'project.dataset.table'."}},
        ["table_ref"]),
    _fn("execute_sql",
        "Run a READ-ONLY SELECT/WITH query and return rows. Be cost-aware: select "
        "only needed columns, filter by date/partition, aggregate in SQL.",
        {"sql": {"type": "string", "description": "A single SELECT or WITH statement."}},
        ["sql"]),
]

DISPATCH_BASE: Dict[str, Callable[[Dict[str, Any]], Any]] = {
    "calc":           lambda a: tool_calc(a["expression"]),
    "list_datasets":  lambda a: tool_list_datasets(),
    "list_tables":    lambda a: tool_list_tables(a["dataset_id"]),
    "get_table_info": lambda a: tool_get_table_info(a["table_ref"]),
    "execute_sql":    lambda a: tool_execute_sql(a["sql"]),
}

# --- expanded exploration tools available to every agent (incl. subagents) ---
DECLS_BASE += [
    _fn("get_table_sample",
        "Return a few sample rows from a table (read-only, cost-guarded). Use after "
        "get_table_info to see real values before writing analytical SQL.",
        {"table_ref": {"type": "string", "description": "'dataset.table' or 'project.dataset.table'."},
         "n": {"type": "integer", "description": "How many rows (1-20, default 5)."}},
        ["table_ref"]),
    _fn("profile_column",
        "Profile one column: row count, distinct values, nulls, min and max in a single "
        "pass. Use to confirm a column's grain/range before trusting it in a metric.",
        {"table_ref": {"type": "string"}, "column": {"type": "string"}},
        ["table_ref", "column"]),
    _fn("dry_run_cost",
        "Estimate how many bytes a SELECT/WITH query would scan WITHOUT running it. "
        "Use to size an expensive query before paying for it.",
        {"sql": {"type": "string", "description": "A single SELECT or WITH statement."}},
        ["sql"]),
]
DISPATCH_BASE.update({
    "get_table_sample": lambda a: tool_get_table_sample(a["table_ref"], a.get("n", 5)),
    "profile_column":   lambda a: tool_profile_column(a["table_ref"], a["column"]),
    "dry_run_cost":     lambda a: tool_dry_run_cost(a["sql"]),
})

log.info(f"Base tool registry: {list(DISPATCH_BASE)}")

## Phase 3 — The tool loop and subagent discipline

The hand-rolled perception→action→observation loop over Gemini's `generate_content`, with the
SDK's automatic function-calling disabled so *we* drive every step. `spawn_subagent` runs a fresh,
isolated `agent_loop` on the faster model with the BASE tools only (no recursion) — used to keep
the lead's context clean during noisy warehouse exploration.

In [ ]:
"""
Agent loop -- perception -> action -> observation, Gemini edition.

  PERCEPTION : generate_content(contents, tools=our declarations)
  ACTION     : the model's reply (role="model") may carry one or more
               function_call parts -> we run each via dispatch.
  OBSERVATION: we append a role="user" Content of function_response parts,
               then loop. Terminates when the model returns no function calls.

We DISABLE the SDK's automatic function calling so WE drive every step -- that
is the whole point of "from scratch". `contents` (a list of types.Content) is
mutated in place so the caller keeps the full transcript.

Gemini 2.5 thinking: with include_thoughts=True, thought parts come back with
part.thought == True. We log them but never feed them back as input.
"""

# every dispatched tool call is recorded here (used by the smoke test to verify
# the agent actually CALLED a tool rather than hallucinating a result).
TOOL_CALLS: List[str] = []


def _wrap_response(r: Any) -> dict:
    return r if isinstance(r, dict) else {"result": str(r)}


def _truncate(s: str, limit: int = MAX_TOOL_OUTPUT) -> str:
    return s if len(s) <= limit else s[:limit] + f"\n... [truncated {len(s)-limit} chars]"


def _dispatch_call(name: str, args: Dict[str, Any], dispatch: Dict[str, Callable]) -> dict:
    preview = ", ".join(f"{k}={str(v)[:50]!r}" for k, v in args.items())
    log_tool.info(f"-> {name}({preview})")
    TOOL_CALLS.append(name)
    if name not in dispatch:
        return {"status": "error", "error": f"Unknown tool {name!r}. Available: {sorted(dispatch)}"}
    try:
        result = dispatch[name](args)
    except Exception as e:
        log_tool.exception(f"tool {name} raised")
        return {"status": "error", "error": f"{type(e).__name__}: {e}"}
    out = _wrap_response(result)
    # protect context: truncate the serialized payload if huge
    blob = json.dumps(out, default=str)
    if len(blob) > MAX_TOOL_OUTPUT:
        out = {"status": out.get("status", "ok"), "truncated": True,
               "preview": _truncate(blob)}
    return out


def _build_config(decls, system_instruction: str) -> "types.GenerateContentConfig":
    return types.GenerateContentConfig(
        system_instruction=system_instruction or None,
        tools=[types.Tool(function_declarations=decls)] if decls else None,
        temperature=0.2,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
        thinking_config=types.ThinkingConfig(
            thinking_budget=THINKING_BUDGET, include_thoughts=True),
    )


def agent_loop(contents: List["types.Content"], decls, dispatch, *,
               model: str = None, system_instruction: str = "",
               max_turns: int = MAX_TURNS, label: str = "lead") -> str:
    model = model or MODELS["lead"]
    cfg = _build_config(decls, system_instruction)
    log_loop.info(f"[{label}] start (model={model}, max_turns={max_turns}, "
                  f"tools={[d.name for d in decls]})")

    final_text = ""
    for turn in range(1, max_turns + 1):
        log_loop.info(f"[{label}] turn {turn}/{max_turns}  turns_in_ctx={len(contents)}")
        resp = gemini_generate(model=model, contents=contents, config=cfg, label=label)

        cand = (resp.candidates or [None])[0]
        if cand is None or cand.content is None:
            log_loop.warning(f"[{label}] empty response on turn {turn}")
            return final_text or "[agent_loop] empty model response"
        contents.append(cand.content)
        parts = cand.content.parts or []

        calls, texts = [], []
        for p in parts:
            if getattr(p, "function_call", None):
                calls.append(p.function_call)
            elif getattr(p, "thought", False) and getattr(p, "text", None):
                log_loop.debug(f"[{label}] thought: {p.text[:200]}")
            elif getattr(p, "text", None):
                texts.append(p.text)

        if not calls:                                   # TERMINATION
            final_text = "".join(texts).strip()
            log_loop.info(f"[{label}] DONE turn {turn} -- no calls. final={final_text[:120]!r}")
            return final_text

        log_loop.info(f"[{label}] turn {turn}: model requested {len(calls)} call(s)")
        fr_parts = []
        for fc in calls:
            args = dict(fc.args) if fc.args else {}
            result = _dispatch_call(fc.name, args, dispatch)
            fr_parts.append(types.Part.from_function_response(name=fc.name, response=result))
        contents.append(types.Content(role="user", parts=fr_parts))

    log_loop.warning(f"[{label}] hit MAX_TURNS={max_turns} without termination.")
    return final_text or "[agent_loop] max turns reached without a terminal response"


log.info("Agent loop ready: agent_loop(contents, decls, dispatch, ...)")

In [ ]:
"""
Subagents -- isolated context for noisy warehouse exploration.

When the lead would pollute its own context (mapping an unfamiliar dataset,
probing many tables to find the right grain) it calls spawn_subagent(prompt).
That runs a FRESH agent_loop with:
  - its own empty contents seeded only with the subtask prompt,
  - the faster subagent model,
  - the BASE tools only (BigQuery + calc) -> no spawn_subagent, so no recursion.
Only the subagent's final summary returns to the lead.
"""

SUBAGENT_SYSTEM = (
    "You are a focused data-analyst subagent with READ-ONLY BigQuery access and a "
    "calc tool. You have ONE self-contained subtask (usually: discover which tables/"
    "columns hold some capital-allocation inputs and report their grain + join keys). "
    "Use list_datasets/list_tables/get_table_info to orient and small probing SELECTs "
    "to confirm. When confident, reply with a concise summary and STOP calling tools -- "
    "that final message is what the lead receives. Do not ask questions; assume and proceed."
)


def run_subagent(prompt: str) -> dict:
    sub_id = uuid.uuid4().hex[:6]
    log_sub.info(f"[sub:{sub_id}] spawn -- {prompt[:120]!r}")
    sub_contents = [types.Content(role="user", parts=[types.Part.from_text(text=prompt)])]
    try:
        result = agent_loop(sub_contents, DECLS_BASE, DISPATCH_BASE,
                            model=MODELS["subagent"], system_instruction=SUBAGENT_SYSTEM,
                            label=f"sub:{sub_id}")
    except Exception as e:
        log_sub.exception(f"[sub:{sub_id}] crashed")
        return {"status": "error", "error": f"{type(e).__name__}: {e}"}
    log_sub.info(f"[sub:{sub_id}] done -- {len(result)} chars")
    return {"status": "ok", "summary": result}


# --- Extend (don't mutate) the base registry for the LEAD agent --------------
DECLS_LEAD: List["types.FunctionDeclaration"] = DECLS_BASE + [
    _fn("spawn_subagent",
        "Delegate a self-contained exploration subtask to a fresh subagent (faster "
        "model, isolated context, read-only BQ + calc). Returns only its final summary.",
        {"prompt": {"type": "string",
                    "description": "Detailed self-contained instructions for the subagent."}},
        ["prompt"]),
]
DISPATCH_LEAD: Dict[str, Callable[[Dict[str, Any]], Any]] = {
    **DISPATCH_BASE,
    "spawn_subagent": lambda a: run_subagent(a["prompt"]),
}

log.info(f"Subagent ready. Lead registry: {list(DISPATCH_LEAD)}")

## Phase 4 — The hardening stack

Patterns that turn one model call into a small, self-checking process — re-pointed from hardening
code to hardening an argument: `architect_editor_solve` (plan the analysis, then execute the plan),
`self_refine` (draft → critique → revise), and `adversarial_probe` (red-team: which assumption, if
wrong, flips the funding decision?). The objective check lives in Phase 5.

In [ ]:
"""
Phase 4 -- the hardening stack (ported from v2, re-pointed at analysis).

Backend-neutral reliability patterns that turn one model call into a small, self-checking
process. In v2 these hardened code; here they harden a capital-allocation argument:
  - architect_editor_solve : plan the analysis, then execute the plan (separates 'what to do'
                             from 'doing it' -- fewer skipped steps).
  - self_refine            : draft -> self-critique -> revise.
  - adversarial_probe      : red-team the recommendation -- what assumption, if wrong, FLIPS
                             the funding decision? (the finance analog of attacking code).
The objective check (the 'tests') lives in Phase 5 as verify_analysis.
"""

ARCHITECT_SYSTEM = (
    "You are the ARCHITECT of a capital-allocation analysis. Do NOT compute yet. Produce a "
    "tight, ordered plan: which inputs are needed (cash flows / revenue / capital / risk "
    "weights), the metrics to compute (NPV/IRR/PI/RAROC/payback), the comparison basis, the "
    "decision rule, and the assumptions to state (hurdle rate, horizon, currency, budget)."
)


def architect_editor_solve(task: str, model: str = None) -> dict:
    """Architect drafts the plan; editor executes it into an answer."""
    model = model or MODELS["lead"]
    plan = think_then_answer(task, system=ARCHITECT_SYSTEM, model=model, max_tokens=1500).answer
    out = think_then_answer(
        f"Follow this analysis plan exactly, computing every figure with explicit arithmetic.\n\n"
        f"PLAN:\n{plan}\n\nTASK:\n{task}", model=model, max_tokens=3000).answer
    return {"plan": plan, "output": out}


def self_refine(task: str, iterations: int = 1, model: str = None) -> dict:
    """draft -> critique -> revise, `iterations` times."""
    model = model or MODELS["subagent"]
    draft = think_then_answer(task, model=model).answer
    for _ in range(max(1, iterations)):
        critique = think_then_answer(
            f"Critique this draft for correctness, missing caveats, uncited hurdle rates, and "
            f"currency/horizon mismatches. Be specific.\n\nDRAFT:\n{draft}\n\nTASK:\n{task}",
            model=model).answer
        draft = think_then_answer(
            f"Revise the draft using the critique. Keep what was right.\n\nDRAFT:\n{draft}\n\n"
            f"CRITIQUE:\n{critique}\n\nTASK:\n{task}", model=model).answer
    return {"final": draft}


def adversarial_probe(task: str, analysis: str, n_max: int = 3) -> list:
    """Find assumptions/data issues that would FLIP the allocation decision."""
    try:
        out = gen_json(
            "Red-team this capital-allocation analysis. List the assumptions or data issues "
            "that, if wrong, would change which units get funded. Output JSON: "
            '{"attacks": [{"scenario": str, "severity": "high"|"medium"|"low"}]}.\n\n'
            f"TASK:\n{task}\n\nANALYSIS:\n{analysis}",
            model=MODELS["lead"], max_tokens=800, label="probe")
        attacks = out.get("attacks", []) if isinstance(out, dict) else []
        return attacks[:n_max]
    except Exception as e:
        log.warning(f"adversarial_probe failed: {e}")
        return []


log.info("Phase 4 hardening ready: architect_editor_solve, self_refine, adversarial_probe")

## Phase 5 — Planning, durable state, and the spec layer

`make_plan` emits a dependency-ordered plan; the SQLite **`TaskDAG`** is the backbone the
orchestrator walks; **`BiTemporalMemory`** keeps facts with validity intervals (supersede, never
delete) and keyword recall. The lighter file-based `todo_*` / `task_*` tools remain for the
agent's own live checklist and cross-session backlog.

The reliability hinge is the **spec layer**. In v2 a pytest contract objectively verified the
agent's *code*; here `verify_analysis` does **independent recomputation** — it re-derives every
metric (NPV/IRR/PI) from each candidate's *cited* cash flows with the trusted `calc` helpers,
compares them to the agent's reported figures within tolerance, and checks ranking consistency and
budget adherence. `analysis_with_checks` is the produce → verify → refine loop built on it.

In [ ]:
"""
Phase 5 -- planning + durable state + the SPEC LAYER (ported & re-pointed from v2).

  make_plan            : a dependency-ordered plan as JSON (Gemini).
  TaskDAG              : a SQLite dependency graph the orchestrator walks (v2's, verbatim).
  BiTemporalMemory     : facts with validity intervals -- supersede, never delete (v2's).
  write_definition_of_done + verify_analysis : the finance analog of v2's pytest contract.

THE SPEC LAYER is the reliability hinge. In v2 a pytest contract objectively verified the
agent's CODE. Here the same idea becomes INDEPENDENT RECOMPUTATION: the contract carries each
candidate's cited inputs (cash-flow stream + hurdle, or the RAROC components), and
verify_analysis re-derives every metric with the trusted calc helpers and compares them to the
figures the agent reported -- within tolerance -- then checks ranking consistency and that the
funded set respects the capital budget. Objective, deterministic, no warehouse needed.
"""
import sqlite3
from dataclasses import dataclass, field


# --- make_plan ---------------------------------------------------------------
@dataclass
class PlanStep:
    step_id: str
    description: str
    depends_on: List[str]
    expected_artifact: str


@dataclass
class Plan:
    goal: str
    steps: List[PlanStep]


def make_plan(goal: str, model: str = None) -> Plan:
    model = model or MODELS["lead"]
    try:
        d = gen_json(
            "Produce a step-by-step, dependency-ordered plan for this capital-allocation task. "
            'Output JSON: {"goal": str, "steps": [{"step_id": "s1", "description": str, '
            '"depends_on": [str], "expected_artifact": str}]}.\n\n' + goal,
            model=model, max_tokens=2000, label="plan")
        steps = [PlanStep(s.get("step_id", f"s{i}"), s.get("description", ""),
                          s.get("depends_on", []), s.get("expected_artifact", ""))
                 for i, s in enumerate(d.get("steps", []))]
        return Plan(goal=d.get("goal", goal), steps=steps)
    except Exception:
        return Plan(goal=goal, steps=[])


# --- TaskDAG (SQLite) -- the orchestration backbone --------------------------
class TaskDAG:
    def __init__(self, db_path):
        self.conn = sqlite3.connect(str(db_path), isolation_level=None)
        self.conn.execute("CREATE TABLE IF NOT EXISTS nodes ("
                          "node_id TEXT PRIMARY KEY, title TEXT, status TEXT, "
                          "attempts INTEGER DEFAULT 0, depends_on TEXT)")

    def add_node(self, node_id, title, depends_on=None):
        self.conn.execute("INSERT OR REPLACE INTO nodes VALUES (?,?,?,?,?)",
                          (node_id, title, "pending", 0, json.dumps(depends_on or [])))

    def all_nodes(self):
        return list(self.conn.execute("SELECT node_id, title, status, attempts FROM nodes"))

    def ready_nodes(self):
        done = {r[0] for r in self.conn.execute("SELECT node_id FROM nodes WHERE status='done'")}
        out = []
        for nid, title, deps in self.conn.execute(
                "SELECT node_id, title, depends_on FROM nodes WHERE status='pending'"):
            if all(d in done for d in json.loads(deps)):
                out.append((nid, title))
        return out

    def set_status(self, node_id, status):
        self.conn.execute("UPDATE nodes SET status=?, attempts=attempts+1 WHERE node_id=?",
                          (status, node_id))


# --- BiTemporalMemory --------------------------------------------------------
class BiTemporalMemory:
    """Facts with validity intervals; superseded facts are invalidated, not deleted."""
    def __init__(self):
        self.records = []

    def store(self, fact, kind="observation", source="agent"):
        rec_id = uuid.uuid4().hex[:8]
        self.records.append({"id": rec_id, "fact": fact, "kind": kind, "source": source,
                             "valid_from": time.time(), "valid_to": None})
        return rec_id

    def invalidate(self, fact_id, reason):
        for r in self.records:
            if r["id"] == fact_id and r["valid_to"] is None:
                r["valid_to"] = time.time(); r["invalidated_reason"] = reason

    def query_valid(self, kind=None):
        return [r for r in self.records
                if r["valid_to"] is None and (kind is None or r["kind"] == kind)]

    def recall(self, query, k=3):
        """Keyword-overlap recall (no embeddings)."""
        q = set(re.findall(r"\w+", query.lower()))
        scored = []
        for r in self.query_valid():
            overlap = len(q & set(re.findall(r"\w+", r["fact"].lower())))
            if overlap:
                scored.append((overlap, r["fact"]))
        scored.sort(reverse=True)
        return [f for _, f in scored[:k]]


# --- the spec layer: definition of done + independent recomputation ----------
def recompute_metrics(cashflows: List[float], hurdle: float) -> dict:
    """The trusted, deterministic metric set for a single cash-flow candidate."""
    m = {"npv": _npv(hurdle, cashflows), "irr": _irr(cashflows),
         "payback": _payback(cashflows), "disc_payback": _disc_payback(hurdle, cashflows)}
    try:
        m["pi"] = _pi(hurdle, cashflows)
    except Exception:
        m["pi"] = None
    return m


def write_definition_of_done(candidates: List[dict], budget: float = None,
                             rank_metric: str = "pi", tolerance: float = 0.01) -> dict:
    """Build (and persist) the analysis contract.

    candidates: [{"id": str, "cashflows": [...], "hurdle": float}] -- the cited inputs.
    budget:     capital budget the funded set must respect (sum of |outlays|), or None.
    rank_metric: which metric the ranking must be sorted by (npv / irr / pi).
    """
    contract = {"candidates": candidates, "budget": budget,
                "rank_metric": rank_metric, "tolerance": tolerance}
    try:
        (WORKDIR / "DEFINITION_OF_DONE.json").write_text(json.dumps(contract, indent=2),
                                                         encoding="utf-8")
    except OSError:
        pass
    return contract


def verify_analysis(contract: dict, findings: dict) -> dict:
    """Independently recompute every metric from the contract's cited inputs and compare to
    the agent's reported `findings` within tolerance; check ranking + budget adherence.

    findings = {"metrics": {id: {"npv":..,"irr":..,"pi":..}}, "ranking": [id,...],
                "funded": [id,...]}
    Returns {"passed": int, "failed": int, "all_passed": bool, "checks": [...]}.
    """
    tol = contract.get("tolerance", 0.01)
    rank_metric = contract.get("rank_metric", "pi")
    checks = []

    def add(name, ok, detail=""):
        checks.append({"name": name, "ok": bool(ok), "detail": detail})

    reported = (findings or {}).get("metrics", {}) or {}
    truth = {}
    for c in contract["candidates"]:
        cid = c["id"]
        t = recompute_metrics(c["cashflows"], c["hurdle"])
        truth[cid] = t
        rep = reported.get(cid, {})
        add(f"{cid}:reported", bool(rep), "no metrics reported for this candidate")
        for metric in ("npv", "irr", "pi"):
            tv = t.get(metric)
            if tv is None:
                continue
            rv = rep.get(metric)
            if rv is None:
                add(f"{cid}:{metric}", False, "metric missing")
                continue
            denom = abs(tv) if abs(tv) > 1e-9 else 1.0
            ok = abs(float(rv) - tv) / denom <= tol
            add(f"{cid}:{metric}", ok, f"reported {rv} vs recomputed {tv:.6g}")

    # ranking must be sorted by the agreed metric, descending, over the truth values
    ranking = (findings or {}).get("ranking", []) or []
    if ranking:
        order_truth = sorted(truth, key=lambda i: (truth[i].get(rank_metric) or float("-inf")),
                             reverse=True)
        add("ranking:consistent", list(ranking) == order_truth,
            f"reported {ranking} vs by-{rank_metric} {order_truth}")

    # funded set must respect the capital budget (sum of initial outlays)
    budget = contract.get("budget")
    funded = (findings or {}).get("funded", []) or []
    if budget is not None and funded:
        outlay_of = {c["id"]: -c["cashflows"][0] for c in contract["candidates"]}
        total = sum(outlay_of.get(i, 0.0) for i in funded)
        add("budget:respected", total <= budget + 1e-6,
            f"funded outlay {total} vs budget {budget}")

    passed = sum(1 for c in checks if c["ok"])
    failed = [c for c in checks if not c["ok"]]
    return {"passed": passed, "failed": len(failed), "all_passed": not failed,
            "checks": checks, "truth": truth}


def analysis_with_checks(task: str, contract: dict, max_rounds: int = 3) -> dict:
    """code_with_tests analog: produce a JSON findings object, verify it by independent
    recomputation, and refine on failure. Returns the findings + verification."""
    feedback = ""
    findings = {}
    cand_ids = [c["id"] for c in contract["candidates"]]
    schema_hint = ('Output JSON: {"metrics": {"<id>": {"npv": num, "irr": num, "pi": num}}, '
                   '"ranking": ["<id>", ...], "funded": ["<id>", ...]}. '
                   f"Candidate ids: {cand_ids}. Rank by {contract.get('rank_metric','pi')}; "
                   f"respect budget {contract.get('budget')} (sum of initial outlays).")
    for rnd in range(1, max_rounds + 1):
        prompt = task + "\n\n" + schema_hint
        if feedback:
            prompt += f"\n\nYour previous answer failed these checks; fix them:\n{feedback}"
        try:
            findings = gen_json(prompt, model=MODELS["lead"], max_tokens=1500, label="analyse")
        except Exception as e:
            feedback = f"could not parse findings JSON: {e}"
            continue
        v = verify_analysis(contract, findings)
        if v["all_passed"]:
            return {"success": True, "rounds": rnd, "findings": findings, "verification": v}
        feedback = "; ".join(f"{c['name']}: {c['detail']}" for c in v["checks"] if not c["ok"])[:600]
    return {"success": False, "rounds": max_rounds, "findings": findings,
            "verification": verify_analysis(contract, findings)}


log.info("Phase 5 ready: make_plan, TaskDAG, BiTemporalMemory, definition_of_done + "
         "verify_analysis (independent recomputation) + analysis_with_checks")

In [ ]:
"""
Todo planning -- the agent's working checklist for the CURRENT request.

Lightweight, ordered scratch paper: written at the start of a multi-step
analysis, ticked off as steps complete, overwritten next request. Persisted to
TODO_FILE so it survives a kernel restart.
"""

_TODO_STATUSES = ("pending", "in_progress", "done")


def _load_todos() -> List[Dict[str, Any]]:
    if not TODO_FILE.exists():
        return []
    try:
        return json.loads(TODO_FILE.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        return []


def _save_todos(items): TODO_FILE.write_text(json.dumps(items, indent=2), encoding="utf-8")


def run_todo_write(items: List[str]) -> dict:
    todos = [{"index": i, "text": t, "status": "pending"} for i, t in enumerate(items)]
    _save_todos(todos)
    log_todo.info(f"[todo_write] {len(todos)} items")
    return {"status": "ok", "todos": run_todo_read()["todos"]}


def run_todo_read() -> dict:
    todos = _load_todos()
    marks = {"pending": "[ ]", "in_progress": "[~]", "done": "[x]"}
    lines = [f"{marks.get(t['status'],'[?]')} {t['index']}: {t['text']}" for t in todos]
    return {"status": "ok", "todos": lines or ["(no todos)"]}


def run_todo_update(index: int, status: str) -> dict:
    if status not in _TODO_STATUSES:
        return {"status": "error", "error": f"status must be one of {_TODO_STATUSES}"}
    todos = _load_todos()
    for t in todos:
        if t["index"] == index:
            t["status"] = status
            _save_todos(todos)
            log_todo.info(f"[todo_update] #{index} -> {status}")
            return {"status": "ok", "todos": run_todo_read()["todos"]}
    return {"status": "error", "error": f"no todo with index {index}"}


DECLS_LEAD += [
    _fn("todo_write",
        "Set the working todo list. Call at the start of any multi-step analysis. "
        "Replaces the existing list.",
        {"items": {"type": "array", "items": {"type": "string"},
                   "description": "Ordered plain-text steps."}}, ["items"]),
    _fn("todo_read", "Show the current todo list with statuses."),
    _fn("todo_update", "Update one todo's status as you progress.",
        {"index": {"type": "integer", "description": "0-based index from todo_write."},
         "status": {"type": "string", "enum": list(_TODO_STATUSES)}}, ["index", "status"]),
]
DISPATCH_LEAD.update({
    "todo_write":  lambda a: run_todo_write(a["items"]),
    "todo_read":   lambda a: run_todo_read(),
    "todo_update": lambda a: run_todo_update(a["index"], a["status"]),
})

log.info(f"Todo tools registered. Lead registry: {list(DISPATCH_LEAD)}")

In [ ]:
"""
Task graph -- a persistent, dependency-aware backlog.

Heavier sibling of the todo list: stable IDs, dependencies (a DAG), priority,
and a result field, persisted to TASKS_FILE. task_next computes the next
UNBLOCKED task (pending, all deps done) -- turning "what now?" into a graph query.
Useful when a capital-allocation study spans several sessions.
"""

_TASK_STATUSES = ("pending", "in_progress", "done", "failed")


def _load_tasks() -> List[Dict[str, Any]]:
    if not TASKS_FILE.exists():
        return []
    try:
        return json.loads(TASKS_FILE.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        return []


def _save_tasks(tasks): TASKS_FILE.write_text(json.dumps(tasks, indent=2), encoding="utf-8")


def run_task_create(description: str, depends_on=None, priority: str = "medium") -> dict:
    tasks = _load_tasks()
    tid = uuid.uuid4().hex[:8]
    tasks.append({"id": tid, "description": description, "status": "pending",
                  "priority": priority, "depends_on": depends_on or [], "result": ""})
    _save_tasks(tasks)
    log.info(f"[task] created {tid}: {description[:60]}")
    return {"status": "ok", "id": tid}


def run_task_list() -> dict:
    tasks = _load_tasks()
    lines = [f"[{t['id']}] {t['status']:11s} {t['priority']:6s}"
             f"{(' needs=' + ','.join(t['depends_on'])) if t['depends_on'] else ''}  {t['description']}"
             for t in tasks]
    return {"status": "ok", "tasks": lines or ["(no tasks)"]}


def run_task_update(task_id: str, status: str, result: str = "") -> dict:
    if status not in _TASK_STATUSES:
        return {"status": "error", "error": f"status must be one of {_TASK_STATUSES}"}
    tasks = _load_tasks()
    for t in tasks:
        if t["id"].startswith(task_id):
            t["status"] = status
            if result:
                t["result"] = result
            _save_tasks(tasks)
            log.info(f"[task] {t['id']} -> {status}")
            return {"status": "ok", "id": t["id"]}
    return {"status": "error", "error": f"no task matching {task_id!r}"}


def run_task_next() -> dict:
    tasks = _load_tasks()
    done = {t["id"] for t in tasks if t["status"] == "done"}
    for t in tasks:
        if t["status"] == "pending" and all(d in done for d in t["depends_on"]):
            return {"status": "ok", "next": f"[{t['id']}] ({t['priority']}) {t['description']}"}
    return {"status": "ok", "next": "No unblocked tasks."}


DECLS_LEAD += [
    _fn("task_create", "Create a persistent task in the dependency graph (heavier than todo).",
        {"description": {"type": "string"},
         "depends_on": {"type": "array", "items": {"type": "string"},
                        "description": "IDs that must be done first."},
         "priority": {"type": "string", "enum": ["high", "medium", "low"]}}, ["description"]),
    _fn("task_list", "List all tasks with IDs, status, priority and dependencies."),
    _fn("task_update", "Update a task's status (and optionally record its result).",
        {"task_id": {"type": "string", "description": "Full ID or unique prefix."},
         "status": {"type": "string", "enum": list(_TASK_STATUSES)},
         "result": {"type": "string"}}, ["task_id", "status"]),
    _fn("task_next", "Ask the graph for the next unblocked task."),
]
DISPATCH_LEAD.update({
    "task_create": lambda a: run_task_create(a["description"], a.get("depends_on"), a.get("priority", "medium")),
    "task_list":   lambda a: run_task_list(),
    "task_update": lambda a: run_task_update(a["task_id"], a["status"], a.get("result", "")),
    "task_next":   lambda a: run_task_next(),
})

log.info(f"Task graph registered. Lead registry: {list(DISPATCH_LEAD)}")

In [ ]:
"""
Skill loading -- lazy, on-demand domain playbooks.

A skill is a folder under SKILLS_DIR with a SKILL.md (e.g. "how we compute RAROC",
"bank capital-allocation conventions"). The lead's system prompt gets only a cheap
one-line index; load_skill(name) pulls one full guide on demand. Degrades to
"(no skills available)" when SKILLS_DIR is empty -- which is fine.
"""

def _skill_dir(name: str) -> Optional[Path]:
    if not name or "/" in name or ".." in name:
        return None
    d = (SKILLS_DIR / name).resolve()
    try:
        d.relative_to(SKILLS_DIR.resolve())
    except ValueError:
        return None
    return d if (d / "SKILL.md").is_file() else None


def run_list_skills() -> dict:
    if not SKILLS_DIR.is_dir():
        return {"status": "ok", "skills": ["(no skills available)"]}
    entries = []
    for child in sorted(SKILLS_DIR.iterdir()):
        md_path = child / "SKILL.md"
        if not md_path.is_file():
            continue
        summary = ""
        for line in md_path.read_text(encoding="utf-8", errors="replace").splitlines():
            if line.strip():
                summary = line.lstrip("# ").strip()
                break
        entries.append(f"- {child.name}: {summary}")
    return {"status": "ok", "skills": entries or ["(no skills available)"]}


def run_load_skill(name: str) -> dict:
    d = _skill_dir(name)
    if d is None:
        return {"status": "error", "error": f"no skill {name!r}", "available": run_list_skills()["skills"]}
    body = (d / "SKILL.md").read_text(encoding="utf-8", errors="replace")
    return {"status": "ok", "skill": name, "body": _truncate(body)}


def skills_index_for_prompt() -> str:
    idx = run_list_skills()["skills"]
    if idx == ["(no skills available)"]:
        return ""
    return ("\n\nAvailable skills (call load_skill(name) for the full guide when relevant):\n"
            + "\n".join(idx))


DECLS_LEAD += [
    _fn("list_skills", "List available skill guides (name + one-line summary)."),
    _fn("load_skill", "Load the full text of one skill guide by name.",
        {"name": {"type": "string", "description": "Skill folder name from list_skills."}}, ["name"]),
]
DISPATCH_LEAD.update({
    "list_skills": lambda a: run_list_skills(),
    "load_skill":  lambda a: run_load_skill(a["name"]),
})

log.info(f"Skill loading registered. Lead registry: {list(DISPATCH_LEAD)}")

## Phase 6 — Context engineering: a bounded working window

A long analysis grows by accumulated **tool output** (schemas, sample rows, query results) more
than by user turns, so the `ContextManager` trims by **model-led steps**: keep the task anchor and
the last N model turns verbatim, distil everything older into `BiTemporalMemory` (`source="distill"`),
and re-inject a compact `<durable_memory>` block into the system instruction. `managed_loop` is
`agent_loop` plus this window — the upgrade over the old summarise-everything compaction, and what
`chat()` now runs on.

In [ ]:
"""
Phase 6 -- context engineering: a bounded working window (ported from v2, Gemini edition).

Supersedes the simple 3-layer compaction. A long analysis grows by accumulated TOOL OUTPUT
(schemas, sample rows, query results) far more than by user turns, so we trim by
MODEL-LED STEPS: keep the task anchor + the last N model turns verbatim, distil everything
older into the Phase-5 BiTemporalMemory (source="distill"), and re-inject a compact
<durable_memory> block into the system instruction. managed_loop is agent_loop + this window.
"""
log_ctx = logging.getLogger("fin.ctx")

CONTEXT_POLICY = (
    "<context_policy>\n"
    "The <durable_memory> block below is distilled from earlier analysis steps that were\n"
    "trimmed to keep your working context small. Treat it as an accurate record of what was\n"
    "already discovered -- tables/columns, SQL run, figures computed, assumptions made -- and\n"
    "do NOT re-query or recompute what it marks as done. Only the most recent steps are shown\n"
    "verbatim; the latest tool results still take precedence over the block.\n"
    "</context_policy>"
)

DISTILL_SYSTEM = (
    "You compress a capital-allocation analyst's tool transcript into durable facts. Output "
    'STRICT JSON: {"facts": [{"fact": str, "kind": str}]}. Keep only HIGH-SIGNAL, reusable '
    "facts: tables/columns discovered and their grain, SQL run and the key figures it "
    "returned, metrics computed (NPV/IRR/RAROC/...), assumptions (hurdle/horizon/currency), "
    "and decisions. Drop pleasantries and raw row dumps. Each fact is one short sentence. "
    "kind in {data, metric, assumption, decision, observation}."
)


@dataclass
class TrimResult:
    contents: list
    trimmed: bool
    dropped: int
    distilled: int


def _content_summary(c) -> str:
    parts = []
    for p in (getattr(c, "parts", None) or []):
        if getattr(p, "function_call", None):
            parts.append(f"(call {p.function_call.name})")
        elif getattr(p, "function_response", None):
            parts.append(f"(result {p.function_response.name})")
        elif getattr(p, "text", None):
            parts.append(p.text)
    return f"{getattr(c, 'role', '?')}: " + " ".join(parts)


class ContextManager:
    """Bounds an analysis loop's working context and preserves what it drops as memory."""

    def __init__(self, memory: "BiTemporalMemory", max_steps: int = 6,
                 model: str = None, recall_k: int = 6, call_timeout=None):
        self.memory = memory
        self.max_steps = max(1, int(max_steps))
        self.model = model or MODELS["summarizer"]
        self.recall_k = recall_k

    def _split(self, contents):
        """anchor = the first user turn (the task); body = everything after."""
        if contents and getattr(contents[0], "role", None) == "user":
            return [contents[0]], list(contents[1:])
        return [], list(contents)

    def trim(self, contents):
        anchor, body = self._split(contents)
        starts = [j for j, c in enumerate(body) if getattr(c, "role", None) == "model"]
        if len(starts) <= self.max_steps:
            return TrimResult(contents, False, 0, 0)
        cut = starts[-self.max_steps]
        dropped, kept = body[:cut], body[cut:]
        n = self._distill(dropped)
        log_ctx.info(f"trim: dropped {len(dropped)} turns "
                     f"({len(starts) - self.max_steps} steps) -> {n} facts; "
                     f"window now anchor + {len(kept)} turns")
        return TrimResult(anchor + kept, True, len(dropped), n)

    def _distill(self, dropped):
        text = "\n".join(_content_summary(c) for c in dropped)
        if not text.strip():
            return 0
        try:
            facts = gen_json("TRANSCRIPT:\n" + text[:12000], model=self.model,
                             system=DISTILL_SYSTEM, max_tokens=1024, label="distill").get("facts", [])
        except Exception as e:
            log_ctx.warning(f"distill failed ({type(e).__name__}: {e}); storing raw summary")
            facts = [{"fact": text[:200], "kind": "observation"}]
        stored = 0
        for f in facts:
            if isinstance(f, dict):
                fact, kind = (f.get("fact") or "").strip(), f.get("kind") or "observation"
            else:
                fact, kind = str(f).strip(), "observation"
            if fact:
                self.memory.store(fact, kind=kind, source="distill")
                stored += 1
        return stored

    def render_block(self, query: str) -> str:
        facts = self.memory.recall(query, k=self.recall_k)
        if not facts:
            return ""
        body = "\n".join(f"- {f}" for f in facts)
        return f"\n\n{CONTEXT_POLICY}\n\n<durable_memory>\n{body}\n</durable_memory>"

    def consolidate(self, critic: bool = True) -> int:
        valid = [r for r in self.memory.query_valid() if r.get("source") == "distill"]
        if len(valid) < 2:
            return len(valid)
        facts_json = json.dumps([{"id": r["id"], "fact": r["fact"], "kind": r["kind"]}
                                 for r in valid])
        try:
            merged = gen_json(
                "Merge an analyst's distilled facts into a clean, de-duplicated set. Drop "
                "redundant/superseded facts; when two conflict keep the later one. Output "
                'STRICT JSON: {"facts": [{"fact": str, "kind": str}]}.\n\nFACTS:\n' + facts_json,
                model=self.model, max_tokens=1024, label="consolidate").get("facts", [])
        except Exception:
            return len(valid)
        if critic and merged:
            try:
                v = gen_json("You audit a memory rewrite. Did the CONSOLIDATED set drop any "
                             'important, non-redundant fact? Output JSON {"safe": bool, "reason": str}.'
                             f"\n\nORIGINAL:\n{facts_json}\n\nCONSOLIDATED:\n{json.dumps(merged)}",
                             model=self.model, max_tokens=512, label="consolidate-critic")
                if not v.get("safe", True):
                    log_ctx.warning(f"consolidate rejected by critic: {v.get('reason')}")
                    return len(valid)
            except Exception:
                pass
        for r in valid:
            self.memory.invalidate(r["id"], reason="consolidated")
        for f in merged:
            fact = (f.get("fact") or "").strip()
            if fact:
                self.memory.store(fact, kind=f.get("kind") or "observation", source="distill")
        log_ctx.info(f"consolidate: {len(valid)} facts -> {len(merged)}")
        return len(merged)


def managed_loop(contents, decls, dispatch, ctx: "ContextManager", *,
                 model: str = None, system_instruction: str = "",
                 max_turns: int = MAX_TURNS, label: str = "lead") -> str:
    """agent_loop + a bounded working window: trim old steps each turn and re-inject the
    distilled <durable_memory> block into the system instruction."""
    model = model or MODELS["lead"]
    base_system = system_instruction
    task = ""
    for c in contents:
        if getattr(c, "role", None) == "user":
            task = " ".join(p.text for p in (c.parts or []) if getattr(p, "text", None))
            break
    log_loop.info(f"[{label}] managed start (model={model}, max_steps={ctx.max_steps})")

    final_text = ""
    for turn in range(1, max_turns + 1):
        res = ctx.trim(contents)
        contents[:] = res.contents
        sys_now = base_system + (ctx.render_block(task) if res.trimmed else "")
        cfg = _build_config(decls, sys_now)
        resp = gemini_generate(model=model, contents=contents, config=cfg, label=label)
        cand = (resp.candidates or [None])[0]
        if cand is None or cand.content is None:
            return final_text or "[managed_loop] empty model response"
        contents.append(cand.content)
        calls, texts = [], []
        for p in (cand.content.parts or []):
            if getattr(p, "function_call", None):
                calls.append(p.function_call)
            elif getattr(p, "thought", False):
                continue
            elif getattr(p, "text", None):
                texts.append(p.text)
        if not calls:
            final_text = "".join(texts).strip()
            log_loop.info(f"[{label}] DONE turn {turn} (no calls)")
            return final_text
        fr_parts = []
        for fc in calls:
            args = dict(fc.args) if fc.args else {}
            result = _dispatch_call(fc.name, args, dispatch)
            fr_parts.append(types.Part.from_function_response(name=fc.name, response=result))
        contents.append(types.Content(role="user", parts=fr_parts))
    log_loop.warning(f"[{label}] hit max_turns={max_turns}")
    return final_text or "[managed_loop] max turns reached"


log.info("Phase 6 context engineering ready: ContextManager (trim/distill/reinject/consolidate) "
         "+ managed_loop")

In [ ]:
"""
System instruction -- the capital-allocation analyst persona & method.

Ported from the ADK notebook. Drives: orient -> plan -> query carefully ->
compute with calc -> rank & recommend -> explain caveats -> cite tables.
"""

INSTRUCTION = """\
You are a senior analyst supporting a CAPITAL ALLOCATION process. You have
direct, READ-ONLY access to a BigQuery data warehouse and a precise
financial-math calculator. Your analysis informs how limited capital is
deployed across competing projects, business units, segments, or assets.

Work like an investment-committee analyst, not a chatbot:

1. ORIENT. When you don't yet know the data, discover it first. Use list_datasets,
   list_tables and get_table_info to inspect schemas BEFORE writing any SQL.
   Never invent table or column names. Identify the tables holding the inputs you
   need: cash flows, revenue/cost, invested capital, asset/segment identifiers,
   dates, and any risk-weighting or cost-of-capital references. For broad or noisy
   exploration, delegate to spawn_subagent so your own context stays clean.

2. PLAN. Call todo_write to lay out the candidate units (projects/segments/assets)
   you will compare, the tables and grain of each, and how you will join them.
   Confirm join keys from the schema, not from assumption.

3. QUERY CAREFULLY. All access is read-only (SELECT/WITH only). Be cost-aware:
   select only needed columns, filter by partition/date, aggregate in SQL rather
   than pulling raw rows. Validate assumptions with small probing queries first.

4. COMPUTE EXACTLY with calc. Never do arithmetic in your head. Per candidate derive:
     - NPV at the relevant discount / hurdle rate   -> calc("npv(...)")
     - IRR of the cash-flow stream                  -> calc("irr(...)")
     - payback period, ROIC / RAROC, profitability index
     - return spread vs. cost of capital (EVA-style)
   Be explicit about the discount / hurdle rate you use and where it came from.

5. RANK & RECOMMEND. Compare candidates on a consistent basis, rank them, and give
   a clear allocation recommendation. Respect any stated capital budget: allocate to
   maximize value within it and note what is funded vs. deferred and why.

6. EXPLAIN BUSINESS MEANING & CAVEATS. Describe what each table represents and the
   business rules you observed (status filters, currency, soft-deletes, capitalized
   vs. expensed items, period cut-offs). Flag data-quality / comparability issues.

7. CITE. Reference the exact project.dataset.table and columns behind every figure.

If a request is ambiguous (discount rate, horizon, or capital budget unspecified),
state the assumption you're making and proceed; surface anything that would
materially change the allocation decision.
"""
print(INSTRUCTION[:200], "...")

In [ ]:
"""
chat() -- the user-facing entry point, now on the BOUNDED working window.

HISTORY is a module-level list of types.Content that persists across calls, so a sequence of
chat() calls is one continuous analysis. Each call appends the user's question and runs
managed_loop with the FULL lead toolset: the ContextManager trims old tool-steps, distils them
into bi-temporal memory, and re-injects a <durable_memory> block -- the Phase 6 upgrade over
the old summarise-everything compaction. reset_chat() wipes the conversation and memory.
"""

HISTORY: List["types.Content"] = []
CHAT_MEMORY = BiTemporalMemory()
CHAT_CTX = ContextManager(CHAT_MEMORY, max_steps=KEEP_RECENT)


def _system_prompt() -> str:
    return INSTRUCTION + skills_index_for_prompt()


def reset_chat() -> None:
    HISTORY.clear()
    CHAT_MEMORY.records.clear()
    try:
        MEMORY_FILE.unlink(missing_ok=True)
    except OSError:
        pass
    log_chat.info("chat history + working memory reset")


def chat(query: str) -> str:
    log_chat.info(f"=== USER: {query[:200]!r} ===")
    HISTORY.append(types.Content(role="user", parts=[types.Part.from_text(text=query)]))
    answer = managed_loop(HISTORY, DECLS_LEAD, DISPATCH_LEAD, CHAT_CTX,
                          model=MODELS["lead"], system_instruction=_system_prompt(), label="lead")
    print("\n" + "=" * 70 + "\nANALYST:\n" + "=" * 70)
    print(answer)
    return answer


log.info("chat() ready (bounded window). Call chat('...'); reset_chat() to start over.")

## Phase 7 — MCP-style registry + the five-subagent analyst architecture

The study becomes a small org with a fixed pipeline over one `TaskDAG` + one `BiTemporalMemory`:
**Planner** (lay out candidates/inputs) → **Analyst** (compute every metric into a structured
findings object, self-correcting against the spec) → **Verifier** (independently re-run
`verify_analysis`) → **Reviewer** (`verifier_score` + `adversarial_probe`) → **ReportWriter**
(the committee memo, via `self_refine`).

In [ ]:
"""
Phase 7 -- MCP-style tool registry + the five-subagent analyst architecture (ported from v2).

A capital-allocation study becomes a small org with a fixed pipeline, each role a subagent
sharing one TaskDAG + one BiTemporalMemory:

  Planner    -> lay out the candidates, inputs and comparison basis (make_plan)
  Analyst    -> compute every metric and produce a structured findings object, self-correcting
                against the spec (analysis_with_checks: produce -> verify -> refine)
  Verifier   -> independently re-run verify_analysis on the findings (the 'tests')
  Reviewer   -> score the analysis + adversarially probe what would flip the decision
  ReportWriter -> write the investment-committee memo (self_refine)
"""

class MCPTool:
    def __init__(self, name, description, handler):
        self.name, self.description, self.handler = name, description, handler

    def execute(self, **kwargs):
        return self.handler(**kwargs)


mcp_registry = {
    "calc":          MCPTool("calc", "Sandboxed finance math", lambda expression: tool_calc(expression)),
    "list_datasets": MCPTool("list_datasets", "List BQ datasets", lambda: tool_list_datasets()),
    "list_tables":   MCPTool("list_tables", "List tables in a dataset", lambda dataset_id: tool_list_tables(dataset_id)),
    "get_table_info":MCPTool("get_table_info", "Table schema + row count", lambda table_ref: tool_get_table_info(table_ref)),
    "get_table_sample": MCPTool("get_table_sample", "Sample rows", lambda table_ref, n=5: tool_get_table_sample(table_ref, n)),
    "execute_sql":   MCPTool("execute_sql", "Read-only SELECT/WITH", lambda sql: tool_execute_sql(sql)),
    "verify_analysis": MCPTool("verify_analysis", "Independent recomputation check",
                               lambda contract, findings: verify_analysis(contract, findings)),
}
log.info(f"MCP registry: {len(mcp_registry)} tools -> {list(mcp_registry)}")


class Subagent:
    def __init__(self, name, parent):
        self.name, self.parent = name, parent


class Planner(Subagent):
    def execute(self, node_id, title):
        plan = make_plan(self.parent.task)
        for s in plan.steps:
            self.parent.memory.store(f"plan step {s.step_id}: {s.description}", kind="decision")
        return {"success": True, "note": f"{len(plan.steps)} plan steps"}


class Analyst(Subagent):
    """The 'implementer': compute all metrics and produce a findings object that passes the
    spec, self-correcting against verify_analysis (architect/editor -> produce -> verify loop)."""
    def execute(self, node_id, title):
        # An architect/editor pass frames the approach, then we drive the checked findings loop.
        framing = architect_editor_solve(self.parent.task)["plan"]
        self.parent.memory.store(f"analysis approach: {framing[:160]}", kind="decision")
        res = analysis_with_checks(self.parent.task, self.parent.contract, max_rounds=3)
        self.parent.findings = res["findings"]
        v = res["verification"]
        for cid, m in (res["findings"].get("metrics", {}) or {}).items():
            self.parent.memory.store(
                f"{cid}: NPV={m.get('npv')}, IRR={m.get('irr')}, PI={m.get('pi')}", kind="metric")
        return {"success": res["success"], "rounds": res["rounds"],
                "passed": v["passed"], "failed": v["failed"]}


class Verifier(Subagent):
    """The 'tester': independently re-run the recomputation contract on the findings."""
    def execute(self, node_id, title):
        v = verify_analysis(self.parent.contract, self.parent.findings or {})
        self.parent.memory.store(
            f"verification: {v['passed']} checks passed, {v['failed']} failed", kind="observation")
        return {"success": v["all_passed"], "passed": v["passed"], "failed": v["failed"],
                "failing": [c["name"] for c in v["checks"] if not c["ok"]][:5]}


class Reviewer(Subagent):
    def execute(self, node_id, title):
        summary = json.dumps(self.parent.findings or {}, default=str)[:2000]
        score = verifier_score(self.parent.task, summary)
        attacks = adversarial_probe(self.parent.task, summary, n_max=3)
        self.parent.memory.store(
            f"review score {score.get('score')}/10: {score.get('reason')}", kind="observation")
        return {"success": True, "score": score.get("score"), "reason": score.get("reason"),
                "n_attacks": len(attacks),
                "top_attack": (attacks[0]["scenario"][:120] if attacks else None)}


class ReportWriter(Subagent):
    def execute(self, node_id, title):
        facts = "\n".join(f"- {f}" for f in self.parent.memory.recall(self.parent.task, k=10))
        draft = self_refine(
            f"Write a concise investment-committee MEMO (<250 words) for this capital-allocation "
            f"task: the ranking, the recommended allocation under the budget, the hurdle rate "
            f"used, and the key caveats.\n\nTASK:\n{self.parent.task}\n\nWHAT WE FOUND:\n{facts}",
            iterations=1)
        try:
            (WORKDIR / "REPORT.md").write_text(draft["final"], encoding="utf-8")
        except OSError:
            pass
        self.parent.report = draft["final"]
        return {"success": True, "artifact": "REPORT.md"}


class AnalystAgent:
    def __init__(self, task, contract, dag, memory):
        self.task, self.contract, self.dag, self.memory = task, contract, dag, memory
        self.findings = {}
        self.report = ""
        self.routing = {
            "sg1": Planner("planner", self),
            "sg2": Analyst("analyst", self),
            "sg3": Verifier("verifier", self),
            "sg4": Reviewer("reviewer", self),
            "sg5": ReportWriter("report_writer", self),
        }


def analyst_run(agent: AnalystAgent, max_iters: int = 10) -> dict:
    runlog = []
    for i in range(max_iters):
        ready = agent.dag.ready_nodes()
        if not ready:
            return {"status": "done", "log": runlog, "iterations": i}
        node_id, title = ready[0]
        sub = agent.routing.get(node_id)
        if sub is None:
            agent.dag.set_status(node_id, "done"); continue
        log.info(f"[run] iter {i+1}: {node_id} ({sub.name}) -- {title}")
        result = sub.execute(node_id, title)
        runlog.append({"iter": i + 1, "node": node_id, "agent": sub.name, "result": result})
        agent.dag.set_status(node_id, "done" if result.get("success") else "failed")
        if not result.get("success"):
            return {"status": "failed", "failed_node": node_id, "log": runlog}
    return {"status": "max_iters", "log": runlog}


log.info("Five-subagent analyst architecture ready: Planner, Analyst, Verifier, Reviewer, ReportWriter")

## Phase 8 — Run it

Two entry points, both against **real** inputs (no synthetic fixtures):

- **`chat(...)`** — the conversational analyst on the bounded window with the full lead toolset and
  skills. The smoke test below exercises the whole stack with a single `calc` question (no
  warehouse needed); point it at your warehouse by asking a real question once
  `bigquery_healthcheck()` passes.
- **The five-subagent pipeline** — `run_allocation_from_warehouse(...)` pulls each candidate's
  cash-flow stream from BigQuery, builds the recomputation contract from those cited inputs, runs
  Planner→Analyst→Verifier→Reviewer→ReportWriter, and independently verifies. It runs only when
  BigQuery is reachable.

In [ ]:
"""
Smoke test -- drive the whole pipeline end to end.

This intentionally needs NO warehouse: it asks a pure-`calc` capital question so
it exercises the full stack -- gemini_healthcheck -> chat -> agent_loop ->
generate_content -> function-calling -> calc dispatch -> terminal text -- and we
verify the SIDE EFFECT (a calc tool call actually happened), not just the prose.

To test BigQuery too, authenticate (see the setup cell), run
bigquery_healthcheck(), then ask a real question, e.g.:

    chat("List datasets and tables, find project cash flows and invested capital, "
         "then NPV at 10% / IRR / payback per project and recommend a 500M allocation.")
"""

assert gemini_healthcheck(), "Gemini not reachable -- check auth / GOOGLE_CLOUD_PROJECT / GEMINI_API_KEY."

reset_chat()
TOOL_CALLS.clear()

answer = chat(
    "A project has cash flows [-1000, 300, 400, 500, 600] over years 0-4. Using the "
    "calc tool, compute its NPV at a 10% hurdle rate and its IRR, then tell me whether "
    "to fund it and why."
)

print("\n" + "-" * 70)
expected_npv = _npv(0.10, [-1000, 300, 400, 500, 600])
print(f"reference NPV@10% = {expected_npv:.2f}   reference IRR = {_irr([-1000,300,400,500,600]):.4f}")
print(f"tools the agent actually called: {TOOL_CALLS}")
ok = "calc" in TOOL_CALLS and bool(answer.strip())
print(f"smoke test : {'PASS' if ok else 'FAIL'}  (agent used calc and returned an answer)")
print("-" * 70)

In [ ]:
# --- Optional: drive the five-subagent pipeline against the REAL warehouse ---------------
# This runs ONLY if BigQuery is reachable; it never fabricates data. Fill in the table/columns
# for your warehouse, then run. The flow is: pull each candidate's ordered cash-flow stream
# from BigQuery -> build the recomputation contract from those CITED inputs -> run the pipeline
# (Planner -> Analyst -> Verifier -> Reviewer -> ReportWriter) -> independently verify.

def run_allocation_from_warehouse(cashflow_sql: str, *, hurdle: float, budget: float,
                                  rank_metric: str = "pi", id_col: str = "project_id",
                                  period_col: str = "period", amount_col: str = "cf") -> dict:
    """cashflow_sql must return one row per (candidate, period) with columns
    [id_col, period_col, amount_col], ordered so t=0 comes first per candidate."""
    res = tool_execute_sql(cashflow_sql)
    if res.get("status") != "ok":
        return {"status": "error", "error": res.get("error"), "sql_result": res}
    streams: Dict[str, List[float]] = {}
    for row in res["rows"]:
        streams.setdefault(str(row[id_col]), []).append(float(row[amount_col]))
    candidates = [{"id": cid, "cashflows": cfs, "hurdle": hurdle}
                  for cid, cfs in streams.items()]
    if not candidates:
        return {"status": "error", "error": "query returned no candidates"}

    contract = write_definition_of_done(candidates, budget=budget, rank_metric=rank_metric)
    task = ("Rank these projects (cash flows pulled from the warehouse) and recommend an "
            f"allocation within a budget of {budget:.0f} at a {hurdle:.0%} hurdle. Cash flows:\n"
            + "\n".join(f"  {c['id']} = {c['cashflows']}" for c in candidates))
    dag = TaskDAG(str(WORKDIR / ".fin_run.db"))
    for nid, title, deps in [("sg1", "Plan", []), ("sg2", "Compute", ["sg1"]),
                             ("sg3", "Verify", ["sg2"]), ("sg4", "Review", ["sg3"]),
                             ("sg5", "Memo", ["sg4"])]:
        dag.add_node(nid, title, deps)
    agent = AnalystAgent(task, contract, dag, BiTemporalMemory())
    run = analyst_run(agent, max_iters=10)
    verification = verify_analysis(contract, agent.findings or {})
    return {"status": "ok", "run": run, "findings": agent.findings,
            "verification": verification, "report": agent.report}


if bigquery_healthcheck():
    print("BigQuery reachable. Edit the SQL below for your warehouse, then call:")
    print("""
    out = run_allocation_from_warehouse(
        '''SELECT project_id, period, SUM(amount) AS cf
           FROM `your_project.your_dataset.project_cashflows`
           WHERE scenario = 'base'
           GROUP BY project_id, period
           ORDER BY project_id, period''',
        hurdle=0.10, budget=500_000_000)
    print(out['verification']['all_passed'], out['report'])
    """)
else:
    print("BigQuery not reachable -- skipping the warehouse pipeline demo. "
          "Authenticate (see the setup cell) and re-run this cell to drive it on real data.")

## Phase 9 — What the extra machinery buys you

Versus `financial_analyst_adk.ipynb` (ADK owns the loop) and `financial_analyst_from_scratch_gemini`'s
earlier shape: a problem-type router, a self-checking hardening stack, a **recomputation spec layer**
that catches a wrong figure before it reaches the committee memo, a bounded working window that keeps
long warehouse explorations in budget, and a five-role pipeline with shared bi-temporal memory.
ADK is the better default for a quick agent; this is what you reach for when an allocation number has
to be *defensible*.

In [ ]:
# A quick structural census of the analyst's engine (no model calls, no warehouse).
census = {
    "base tools (subagents)": list(DISPATCH_BASE),
    "lead tools":             list(DISPATCH_LEAD),
    "MCP registry tools":     list(mcp_registry),
    "subagent roles":         ["planner", "analyst", "verifier", "reviewer", "report_writer"],
    "problem router":         f"classify_problem -> {PROBLEM_TYPES} -> strategy",
    "hardening patterns":     ["adaptive_think", "self_consistency", "asymmetric_solve",
                               "architect_editor_solve", "self_refine", "adversarial_probe"],
    "spec layer":             "verify_analysis: independent recomputation of NPV/IRR/PI + "
                              "ranking + budget adherence",
    "working context":        "managed_loop: trim -> distill -> reinject -> consolidate",
    "skills available":       run_list_skills()["skills"],
}
import pprint
pprint.pp(census)

### Where to take it next

- **Extend the spec layer to RAROC.** `verify_analysis` recomputes cash-flow metrics today; add a
  risk-weighted branch (`expected_loss`, `raroc`, `economic_profit`) so lending-book allocations are
  verified the same way. The `raroc` skill already documents the inputs.
- **Tune the working window.** `ContextManager(max_steps=...)` sets how many model turns stay
  verbatim; call `CHAT_CTX.consolidate()` between studies to de-dupe and age the durable memory.
- **Add skills.** Drop a new `SKILL.md` under `fin_sandbox/skills/` (e.g. WACC, EVA, real options)
  and it appears in the prompt index automatically.
- **Carry the structure onward.** The substrate, hardening, spec layer and context engine are
  domain-neutral — the knowledge-management notebooks are the next port.